# Moving Horizon Estimation (MHE): полная теория с априорной информацией и матрицей Фишера

## 1. Постановка задачи

Рассмотрим нелинейную динамическую систему с дискретным временем:

$$
\begin{aligned}
x_{k+1} &= f(x_k, \theta) + w_k, \quad w_k \sim \mathcal{N}(0, Q), \\
y_k &= h(x_k, \theta) + v_k, \quad v_k \sim \mathcal{N}(0, R),
\end{aligned}
$$

где:
- $x_k \in \mathbb{R}^n$ — состояние,
- $\theta \in \mathbb{R}^p$ — неизвестные постоянные параметры,
- $w_k$, $v_k$ — независимые гауссовские шумы процесса и измерений.

Априорная информация о параметрах задаётся распределением $\theta \sim \mathcal{N}(\bar\theta, P_\theta)$.

**Цель MHE:** на каждом временном шаге $k$ оценить $\theta$ и состояние $x_k$ по последним $M$ измерениям, используя всю предыдущую информацию, свёрнутую в априорные слагаемые.

## 2. Байесовский вывод

Апостериорная плотность всех переменных (при фиксированной истории измерений $y_{1:N}$) пропорциональна:

$$
p(\theta, x_{0:N} \mid y_{1:N}) \propto p(\theta) \prod_{k=0}^{N-1} p(x_{k+1}\mid x_k,\theta) \prod_{k=1}^{N} p(y_k\mid x_k,\theta).
$$

Логарифм апостериорной плотности (с точностью до константы):

$$
-\log p = \underbrace{\frac12 \|\theta - \bar\theta\|_{P_\theta^{-1}}^2}_{\text{априорная информация о параметрах}} +
\frac12 \sum_{k=0}^{N-1} \|x_{k+1} - f(x_k,\theta)\|_{Q^{-1}}^2 +
\frac12 \sum_{k=1}^{N} \|y_k - h(x_k,\theta)\|_{R^{-1}}^2 + \text{const}.
$$

## 3. Moving Horizon Estimation (скользящее окно)

В MHE на каждом шаге $k$ решается задача на окне длины $M$, содержащем измерения $y_{k-M+1},\dots,y_k$. При этом априорная информация о параметрах и состоянии в начале окна сохраняется через **arrival cost**.

Оптимизационная задача в момент $k$:

$$
\min_{\theta, x_{k-M+1}} \; \frac12 \|\theta - \hat\theta_{k-1}\|_{C_{\theta,k-1}^{-1}}^2 +
\frac12 \|x_{k-M+1} - \hat x_{k-M+1|k-1}\|_{P_{k-M+1|k-1}^{-1}}^2 +
\frac12 \sum_{i=k-M+1}^{k} \|y_i - h(x_i,\theta)\|_{R^{-1}}^2,
$$

при этом состояния $x_i$ связаны динамикой (или условиями непрерывности при multiple shooting). Здесь:
- $\hat\theta_{k-1}$, $C_{\theta,k-1}$ — оценка параметров и её ковариация после обработки предыдущего окна,
- $\hat x_{k-M+1|k-1}$, $P_{k-M+1|k-1}$ — априорная оценка состояния в начале окна (полученная из предыдущего решения или фильтра).

## 4. Априорная информация о параметрах и её обновление

Параметры $\theta$ постоянны, поэтому априорная ковариация $C_{\theta,k-1}$ должна **уменьшаться** с каждым новым окном (накапливается информация). Обновление производится по правилу сложения матриц Фишера:

$$
C_{\theta,k}^{-1} = C_{\theta,k-1}^{-1} + J_{\theta,k}^T R^{-1} J_{\theta,k},
$$

где $J_{\theta,k}$ — якобиан измерений на текущем окне по параметрам. Аналогично обновляется оценка параметров:

$$
\hat\theta_k = \hat\theta_{k-1} + C_{\theta,k} J_{\theta,k}^T R^{-1} \bigl( y_k - h(\hat x_{k|k-1}, \hat\theta_{k-1}) \bigr).
$$

Это напоминает **фильтр Калмана для параметров**.

## 5. Матрица Фишера (FIM) и её роль

**Определение:** FIM для параметров $\theta$ при наличии данных $y_{1:N}$ есть

$$
\mathcal{I}(\theta) = \mathbb{E}\left[ \left( \frac{\partial \log p(y_{1:N}\mid\theta)}{\partial \theta} \right)^{\!2} \right] = \sum_{k=1}^{N} \left( \frac{\partial h}{\partial \theta} \right)^T R^{-1} \left( \frac{\partial h}{\partial \theta} \right) + \sum_{k=0}^{N-1} \left( \frac{\partial f}{\partial \theta} \right)^T Q^{-1} \left( \frac{\partial f}{\partial \theta} \right) + P_\theta^{-1}.
$$

**Свойства:**
1. Обратная FIM даёт **нижнюю границу Крамера–Рао** для ковариации любой несмещённой оценки:
   $$
   \operatorname{Cov}(\hat\theta) \ge \mathcal{I}(\theta)^{-1}.
   $$
2. В методе Гаусса–Ньютона аппроксимированная матрица Гессе целевой функции есть $J_{\text{full}}^T J_{\text{full}}$, которая при $N\to\infty$ сходится к $\mathcal{I}(\theta)$.
3. FIM аддитивна по данным: информация от независимых измерений складывается.

В MHE матрица $C_{\theta,k-1}^{-1}$ — это накопленная FIM от всех предыдущих окон. Поэтому её обновление через добавление информации нового окна соответствует точному байесовскому выводу.

## 6. Пример: априорная информация как регуляризация

Если положить $\bar\theta = 0$ и $P_\theta = \frac{1}{\lambda} I$, то априорное слагаемое превращается в $\frac{\lambda}{2}\|\theta\|^2$, что соответствует **гребневой регрессии** (ridge regression). В вашем коде вы используете `lambda_reg * I_reg` — это частный случай, но с той разницей, что регуляризация применяется также к начальным условиям шутов. Для настоящего MHE нужно разделить априорную информацию о параметрах и о начальных состояниях.

## 7. Вычисление ковариации оценённых параметров после идентификации

После завершения глобальной идентификации (или после каждого окна MHE) ковариация параметров вычисляется как:

$$
\hat C_\theta = \hat\sigma^2 \left( J_{\text{full}}^T J_{\text{full}} \right)^{-1}_{:p, :p},
$$

где $\hat\sigma^2 = \frac{1}{N_{\text{res}} - p} \|R_{\text{full}}\|^2$ — оценка дисперсии шума, а $J_{\text{full}}$ — полный якобиан всех невязок. Эту ковариацию можно использовать как априорную для следующего окна в рекурсивном режиме.

## 8. Преимущества MHE с априорной информацией о параметрах

1. **Устойчивость к малым выборкам:** априорное слагаемое предотвращает переобучение, когда данных мало.
2. **Рекурсивное обновление:** параметры могут отслеживать медленные изменения во времени.
3. **Естественная регуляризация:** байесовский подход даёт оптимальные веса регуляризации.
4. **Количественная оценка неопределённости:** FIM предоставляет ковариацию оценок.

## 9. Заключение

Moving Horizon Estimation — это мощный метод для совместного оценивания состояния и параметров нелинейных систем. Ключевые элементы:
- **Априорная информация о параметрах** (накопленная FIM),
- **Arrival cost** для состояния,
- **Скользящее окно** для ограничения вычислительной сложности.

Ваш текущий код на основе multiple shooting и Гаусса–Ньютона уже реализует ядро MHE. Добавление априорных слагаемых и рекурсивного обновления ковариации превратит его в полноценный MHE-оценщик, способный работать в реальном времени.

## Литература

1. Rao, C. V., Rawlings, J. B., & Mayne, D. Q. (2003). Constrained state estimation for nonlinear discrete-time systems: stability and moving horizon approximations. *IEEE TAC*, 48(2), 246–258.
2. Rawlings, J. B., & Mayne, D. Q. (2009). *Model Predictive Control: Theory and Design*. Nob Hill Publishing.
3. Haseltine, E. L., & Rawlings, J. B. (2005). Critical evaluation of extended Kalman filtering and moving-horizon estimation. *I&EC Research*, 44(8), 2451–2460.
4. Kay, S. M. (1993). *Fundamentals of Statistical Signal Processing: Estimation Theory*. Prentice-Hall.